# 🔍 Anomaly Detection — End-to-End Machine Learning Pipeline

---

This notebook builds a complete, production-quality anomaly detection system. It dynamically adapts to the uploaded dataset, covering data ingestion, preprocessing, EDA, model training, evaluation, comparison, and business insights.

| Section | Description |
|---|---|
| 1 | Dataset Loading & Overview |
| 2 | Data Preprocessing |
| 3 | Exploratory Data Analysis |
| 4 | Isolation Forest Model |
| 5 | Anomaly Visualization |
| 6 | Evaluation |
| 7 | Algorithm Comparison |
| 8 | PCA-Based Visualization |
| 9 | Contamination Analysis |
| 10 | Anomaly Score Ranking |
| 11 | Feature-Level Explanation |
| 12 | Business Insights Report |
| 13 | Interactive Threshold Experiment |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import ipywidgets as widgets
from IPython.display import display, Markdown

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

PALETTE = {'normal': '#4C72B0', 'anomaly': '#DD4949'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
## 1 · Dataset Loading and Overview

In [ ]:
FILE_PATH = 'creditcard.csv'

df_raw = pd.read_csv(FILE_PATH)

print(f"{'='*55}")
print(f"  Dataset: {FILE_PATH}")
print(f"{'='*55}")
print(f"  Rows        : {df_raw.shape[0]:,}")
print(f"  Columns     : {df_raw.shape[1]}")
print(f"  Missing vals: {df_raw.isnull().sum().sum()}")
print(f"  Duplicates  : {df_raw.duplicated().sum():,}")
print(f"{'='*55}")

In [ ]:
print("Column types:\n")
print(df_raw.dtypes.to_string())

In [ ]:
print("Missing values per column:\n")
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found.")

In [ ]:
df_raw.head()

In [ ]:
df_raw.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std', 'min', 'max'])

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_raw.select_dtypes(exclude=[np.number]).columns.tolist()

def detect_label_column(df, numeric_cols):
    candidates = [c for c in numeric_cols if df[c].nunique() <= 10 and df[c].nunique() >= 2]
    for name_hint in ['class', 'label', 'target', 'fraud', 'anomaly', 'y']:
        for c in candidates:
            if name_hint in c.lower():
                return c
    return None

def detect_id_column(df):
    for c in df.columns:
        if c.lower() in ['id', 'index', 'record_id', 'row_id', 'customerid', 'transaction_id', 'time']:
            return c
    if df.index.name:
        return df.index.name
    return None

LABEL_COL = detect_label_column(df_raw, numeric_cols)
ID_COL = detect_id_column(df_raw)

print(f"Detected label column  : {LABEL_COL}")
print(f"Detected ID column     : {ID_COL}")
print(f"Numeric feature columns: {len(numeric_cols)}")
print(f"Categorical columns    : {categorical_cols if categorical_cols else 'None'}")

In [ ]:
if LABEL_COL:
    vc = df_raw[LABEL_COL].value_counts()
    print(f"Label distribution ({LABEL_COL}):")
    for val, cnt in vc.items():
        print(f"  {val}: {cnt:,}  ({cnt/len(df_raw)*100:.3f}%)")

---
## 2 · Data Preprocessing

In [ ]:
df = df_raw.copy()

before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before_dedup - len(df):,} duplicate rows  →  {len(df):,} rows remain.")

In [ ]:
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(exclude=[np.number]).columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f"Remaining missing values: {df.isnull().sum().sum()}")

In [ ]:
exclude_from_features = []
if LABEL_COL and LABEL_COL in df.columns:
    exclude_from_features.append(LABEL_COL)
    y_true = df[LABEL_COL].values
else:
    y_true = None

if ID_COL and ID_COL in df.columns and ID_COL != LABEL_COL:
    exclude_from_features.append(ID_COL)

feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_from_features]

X_raw = df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Feature matrix shape : {X_scaled.shape}")
print(f"Features used        : {feature_cols}")

---
## 3 · Exploratory Data Analysis

In [ ]:
plot_cols = feature_cols[:min(9, len(feature_cols))]
n = len(plot_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    axes[i].hist(df[col].dropna(), bins=50, color=PALETTE['normal'], edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=PALETTE['normal'], alpha=0.7),
                    medianprops=dict(color='white', linewidth=2))
    axes[i].set_title(f'Boxplot: {col}')
    axes[i].set_ylabel(col)
    axes[i].set_xticks([])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Boxplots', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = feature_cols[:min(15, len(feature_cols))]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap='coolwarm',
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.3,
    cbar_kws={'shrink': 0.7}, ax=ax
)
ax.set_title('Correlation Heatmap (first 15 numeric features)', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
pair_cols = feature_cols[:min(5, len(feature_cols))]
pair_data = df[pair_cols].copy()
if LABEL_COL:
    pair_data['_label'] = df[LABEL_COL].map({0: 'Normal', 1: 'Anomaly'})
    g = sns.pairplot(pair_data, hue='_label', palette={'Normal': PALETTE['normal'], 'Anomaly': PALETTE['anomaly']},
                     plot_kws={'alpha': 0.3, 's': 10}, diag_kind='kde')
else:
    g = sns.pairplot(pair_data, plot_kws={'alpha': 0.3, 's': 10, 'color': PALETTE['normal']}, diag_kind='kde')

g.fig.suptitle('Pair Plot — Selected Features', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---
## 4 · Isolation Forest Model

In [ ]:
CONTAMINATION = 0.01

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

iso_labels_raw = iso_forest.fit_predict(X_scaled)
iso_scores    = iso_forest.decision_function(X_scaled)

iso_labels = (iso_labels_raw == -1).astype(int)

n_anomalies = iso_labels.sum()
pct_anomalies = n_anomalies / len(iso_labels) * 100

print(f"{'='*45}")
print(f"  Isolation Forest Results")
print(f"{'='*45}")
print(f"  Total records  : {len(iso_labels):,}")
print(f"  Anomalies (1)  : {n_anomalies:,}")
print(f"  Normal (0)     : {(iso_labels==0).sum():,}")
print(f"  Anomaly rate   : {pct_anomalies:.2f}%")
print(f"{'='*45}")

In [ ]:
anomaly_indices = np.where(iso_labels == 1)[0]

if ID_COL and ID_COL in df.columns:
    anomaly_ids = df.loc[anomaly_indices, ID_COL].values
    print(f"Anomalous record IDs (by {ID_COL}):")
    print(anomaly_ids[:30], '...' if len(anomaly_ids) > 30 else '')
else:
    print(f"Anomalous record indices (first 30): {anomaly_indices[:30]}")

---
## 5 · Anomaly Visualization

In [ ]:
pca_vis = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_vis.fit_transform(X_scaled)
var_explained = pca_vis.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for is_anomaly, label, color in [(0, 'Normal', PALETTE['normal']), (1, 'Anomaly', PALETTE['anomaly'])]:
    mask = iso_labels == is_anomaly
    alpha = 0.15 if is_anomaly == 0 else 0.8
    size = 8 if is_anomaly == 0 else 30
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, s=size, alpha=alpha, label=label, zorder=is_anomaly+1)

axes[0].set_title('PCA 2D — Isolation Forest Anomalies', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var_explained[0]:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({var_explained[1]:.1f}% var)')
axes[0].legend(frameon=True)

sc = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=iso_scores, cmap='RdYlGn', s=8, alpha=0.5)
plt.colorbar(sc, ax=axes[1], label='Anomaly Score (higher = more normal)')
axes[1].set_title('PCA 2D — Anomaly Score Heatmap', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({var_explained[0]:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({var_explained[1]:.1f}% var)')

plt.tight_layout()
plt.show()

In [ ]:
box_cols = feature_cols[:min(6, len(feature_cols))]
n_box = len(box_cols)
fig, axes = plt.subplots(2, n_box, figsize=(4 * n_box, 8))

for i, col in enumerate(box_cols):
    normal_data = df.loc[iso_labels == 0, col]
    anomaly_data = df.loc[iso_labels == 1, col]

    axes[0, i].boxplot([df[col]], patch_artist=True,
                       boxprops=dict(facecolor=PALETTE['normal'], alpha=0.7),
                       medianprops=dict(color='white', linewidth=2))
    axes[0, i].set_title(col, fontsize=9)
    axes[0, i].set_ylabel('Value')
    axes[0, i].set_xticks([])

    bp = axes[1, i].boxplot(
        [normal_data, anomaly_data],
        patch_artist=True,
        labels=['Normal', 'Anomaly'],
        medianprops=dict(color='white', linewidth=2)
    )
    bp['boxes'][0].set_facecolor(PALETTE['normal'])
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(PALETTE['anomaly'])
    bp['boxes'][1].set_alpha(0.7)
    axes[1, i].set_title(col, fontsize=9)

axes[0, 0].set_ylabel('Before Detection')
axes[1, 0].set_ylabel('After Detection')

plt.suptitle('Feature Boxplots — Before vs After Anomaly Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6 · Evaluation

In [ ]:
if y_true is not None:
    unique_true = np.unique(y_true)
    if set(unique_true).issubset({0, 1}):
        print("Ground-truth labels detected → Supervised Evaluation")
        print("\nClassification Report (Isolation Forest):")
        print(classification_report(y_true, iso_labels, target_names=['Normal', 'Anomaly']))

        p  = precision_score(y_true, iso_labels, zero_division=0)
        r  = recall_score(y_true, iso_labels, zero_division=0)
        f1 = f1_score(y_true, iso_labels, zero_division=0)
        print(f"Precision : {p:.4f}")
        print(f"Recall    : {r:.4f}")
        print(f"F1 Score  : {f1:.4f}")

        cm = confusion_matrix(y_true, iso_labels)
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomaly']).plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title('Confusion Matrix — Isolation Forest', fontweight='bold', pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print("Label column found but contains more than 2 classes — skipping binary evaluation.")
        y_true = None
else:
    display(Markdown("""
**Unsupervised Mode — No Ground-Truth Labels**

No label column was detected in this dataset. Evaluation is performed through:
- Anomaly score ranking (Section 10)
- PCA-based visualizations (Sections 5 & 8)
- Feature-level explanation for flagged records (Section 11)
"""))

---
## 7 · Algorithm Comparison

In [ ]:
N_SAMPLE = min(10_000, len(X_scaled))
sample_idx = np.random.choice(len(X_scaled), N_SAMPLE, replace=False)
X_sample = X_scaled[sample_idx]
y_sample = y_true[sample_idx] if y_true is not None else None

algorithms = {
    'Isolation Forest': IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1),
    'Local Outlier Factor': LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION, n_jobs=-1),
    'One-Class SVM': OneClassSVM(nu=CONTAMINATION, kernel='rbf', gamma='scale'),
}

results = []
algo_labels = {}

for name, model in algorithms.items():
    if isinstance(model, LocalOutlierFactor):
        raw_pred = model.fit_predict(X_sample)
    else:
        raw_pred = model.fit(X_sample).predict(X_sample)

    labels = (raw_pred == -1).astype(int)
    algo_labels[name] = labels
    n_anom = labels.sum()
    pct = n_anom / len(labels) * 100

    row = {'Algorithm': name, 'Anomalies': n_anom, 'Anomaly %': f'{pct:.2f}%'}

    if y_sample is not None and set(np.unique(y_sample)).issubset({0, 1}):
        row['Precision'] = round(precision_score(y_sample, labels, zero_division=0), 4)
        row['Recall']    = round(recall_score(y_sample, labels, zero_division=0), 4)
        row['F1 Score']  = round(f1_score(y_sample, labels, zero_division=0), 4)

    results.append(row)

results_df = pd.DataFrame(results)
results_df

In [ ]:
metric_cols = [c for c in ['Precision', 'Recall', 'F1 Score'] if c in results_df.columns]

if metric_cols:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(results_df))
    width = 0.25
    colors = ['#4C72B0', '#55A868', '#C44E52']
    for i, metric in enumerate(metric_cols):
        ax.bar(x + i * width, results_df[metric], width, label=metric, color=colors[i], alpha=0.85)
    ax.set_xticks(x + width)
    ax.set_xticklabels(results_df['Algorithm'], rotation=10)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.set_title('Algorithm Comparison — Precision / Recall / F1', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(results_df['Algorithm'], results_df['Anomalies'], color=[PALETTE['anomaly'], '#E88C30', '#9B59B6'], alpha=0.85)
    ax.set_ylabel('Anomalies Detected')
    ax.set_title('Anomalies Detected per Algorithm (Unsupervised)', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 8 · PCA-Based Visualization

In [ ]:
pca_full = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca_full.fit_transform(X_scaled)
var = pca_full.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = list(algorithms.keys())

for ax, (algo_name, labels_sample) in zip(axes, algo_labels.items()):
    X_pca_sample = X_pca[sample_idx]
    for lbl, color, label_str, alpha, size in [
        (0, PALETTE['normal'], 'Normal', 0.15, 8),
        (1, PALETTE['anomaly'], 'Anomaly', 0.7, 25),
    ]:
        m = labels_sample == lbl
        ax.scatter(X_pca_sample[m, 0], X_pca_sample[m, 1], c=color, s=size, alpha=alpha, label=label_str)
    ax.set_title(algo_name, fontweight='bold')
    ax.set_xlabel(f'PC1 ({var[0]:.1f}%)')
    ax.set_ylabel(f'PC2 ({var[1]:.1f}%)')
    ax.legend(frameon=True, fontsize=9)

plt.suptitle('PCA 2D — Anomalies Detected by Each Algorithm', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9 · Contamination Analysis

In [ ]:
contamination_values = [0.01, 0.03, 0.05, 0.10]
cont_results = []

for c in contamination_values:
    model = IsolationForest(n_estimators=200, contamination=c, random_state=RANDOM_STATE, n_jobs=-1)
    preds = model.fit_predict(X_scaled)
    labels = (preds == -1).astype(int)
    n_anom = labels.sum()
    row = {'Contamination': c, 'Anomalies': n_anom, 'Anomaly %': round(n_anom / len(labels) * 100, 2)}
    if y_true is not None and set(np.unique(y_true)).issubset({0, 1}):
        row['Precision'] = round(precision_score(y_true, labels, zero_division=0), 4)
        row['Recall']    = round(recall_score(y_true, labels, zero_division=0), 4)
        row['F1 Score']  = round(f1_score(y_true, labels, zero_division=0), 4)
    cont_results.append(row)

cont_df = pd.DataFrame(cont_results)
cont_df

In [ ]:
metric_cols_cont = [c for c in ['Precision', 'Recall', 'F1 Score'] if c in cont_df.columns]
n_subplots = 1 + (1 if metric_cols_cont else 0)

fig, axes = plt.subplots(1, n_subplots, figsize=(7 * n_subplots, 5))
if n_subplots == 1:
    axes = [axes]

axes[0].bar([str(c) for c in cont_df['Contamination']], cont_df['Anomalies'],
            color=PALETTE['anomaly'], alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Contamination Rate')
axes[0].set_ylabel('Anomalies Detected')
axes[0].set_title('Anomaly Count vs Contamination Rate', fontweight='bold')

if metric_cols_cont:
    for metric, color in zip(metric_cols_cont, ['#4C72B0', '#55A868', '#C44E52']):
        axes[1].plot([str(c) for c in cont_df['Contamination']], cont_df[metric],
                     marker='o', linewidth=2, label=metric, color=color)
    axes[1].set_xlabel('Contamination Rate')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Evaluation Metrics vs Contamination Rate', fontweight='bold')
    axes[1].legend()
    axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## 10 · Anomaly Score Ranking

In [ ]:
score_df = df.copy()
score_df['_anomaly_score'] = -iso_forest.decision_function(X_scaled)
score_df['_iso_label'] = iso_labels

top_anomalies = (
    score_df[score_df['_iso_label'] == 1]
    .sort_values('_anomaly_score', ascending=False)
    .reset_index(drop=True)
)

top_anomalies.index = top_anomalies.index + 1
top_anomalies.index.name = 'Rank'

display_cols = []
if ID_COL and ID_COL in top_anomalies.columns:
    display_cols.append(ID_COL)
display_cols += feature_cols[:6]
display_cols.append('_anomaly_score')

top_anomalies[display_cols].head(20).style.background_gradient(cmap='Reds', subset=['_anomaly_score'])

---
## 11 · Explanation of Detected Anomalies

In [ ]:
feature_means = df[feature_cols].mean()
feature_stds  = df[feature_cols].std().replace(0, 1)

TOP_N = 10

top_n_anomalies = top_anomalies.head(TOP_N)

explanation_rows = []

for rank, (_, row) in enumerate(top_n_anomalies.iterrows(), 1):
    rec = {f: row[f] for f in feature_cols}
    z_scores = {f: (rec[f] - feature_means[f]) / feature_stds[f] for f in feature_cols}
    sorted_z = sorted(z_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    top3 = sorted_z[:3]
    explanation = "; ".join(
        f"{feat} is {'HIGH' if z > 0 else 'LOW'} (z={z:+.2f})"
        for feat, z in top3
    )
    explanation_rows.append({
        'Rank': rank,
        'Anomaly Score': round(row['_anomaly_score'], 4),
        'Top Deviant Features': explanation,
    })

exp_df = pd.DataFrame(explanation_rows)
exp_df.set_index('Rank', inplace=True)
exp_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

all_feature_vals = top_n_anomalies[feature_cols].copy()
z_matrix = (all_feature_vals - feature_means) / feature_stds

plot_z = z_matrix[feature_cols[:min(12, len(feature_cols))]].values
plot_feats = feature_cols[:min(12, len(feature_cols))]

im = ax.imshow(plot_z, aspect='auto', cmap='RdBu_r', vmin=-4, vmax=4)
ax.set_xticks(range(len(plot_feats)))
ax.set_xticklabels(plot_feats, rotation=45, ha='right')
ax.set_yticks(range(TOP_N))
ax.set_yticklabels([f'Rank {i+1}' for i in range(TOP_N)])
ax.set_title('Z-Score Heatmap — Top Anomalies vs Feature Means', fontweight='bold')
plt.colorbar(im, ax=ax, label='Z-Score (deviation from mean)')
plt.tight_layout()
plt.show()

---
## 12 · Business Insights Report

In [ ]:
n_total   = len(iso_labels)
n_anom    = int(iso_labels.sum())
pct_anom  = n_anom / n_total * 100
top_deviant = exp_df['Top Deviant Features'].iloc[0] if len(exp_df) > 0 else 'N/A'

report_lines = [
    "# 📋 Business Insights Report",
    "",
    f"**Dataset:** `{FILE_PATH}`",
    f"**Total Records Analyzed:** {n_total:,}",
    f"**Anomalies Detected:** {n_anom:,} ({pct_anom:.2f}%)",
    f"**Detection Method:** Isolation Forest (contamination={CONTAMINATION})",
    "",
    "## Key Findings",
    "",
    f"- Out of **{n_total:,}** records, **{n_anom:,}** were flagged as anomalous.",
    f"- The anomaly rate is **{pct_anom:.2f}%**, which is {'very low (potential high-stakes fraud/errors)' if pct_anom < 2 else 'moderate — warrants further investigation' if pct_anom < 5 else 'elevated — review data quality and contamination setting'}.",
    f"- The highest-ranked anomaly deviates primarily on: `{top_deviant}`.",
    "",
    "## Possible Causes",
    "",
    "- **Fraud or malicious activity:** Unusual feature combinations may indicate fraudulent transactions or intrusion events.",
    "- **Data entry errors:** Some outliers may be clerical mistakes or sensor/system glitches.",
    "- **Genuine rare events:** Anomalies may represent rare-but-valid scenarios (large transactions, extreme usage patterns).",
    "- **Concept drift:** If the model was trained on historical data, distribution shifts over time may cause newer patterns to appear anomalous.",
    "",
    "## Recommended Actions",
    "",
    "1. **Prioritize the top-ranked anomalies** (Section 10) for immediate manual review.",
    "2. **Cross-reference with domain knowledge** — some anomalies may be explainable by business events.",
    "3. **Tune the contamination rate** (Section 13) based on acceptable false-positive rates in the business context.",
    "4. **Monitor over time** — re-run this pipeline periodically to catch emerging anomaly patterns.",
    "",
]

if y_true is not None and set(np.unique(y_true)).issubset({0,1}):
    p_val  = precision_score(y_true, iso_labels, zero_division=0)
    r_val  = recall_score(y_true, iso_labels, zero_division=0)
    f1_val = f1_score(y_true, iso_labels, zero_division=0)
    report_lines += [
        "## Model Performance (Supervised)",
        "",
        f"| Metric | Score |",
        f"|---|---|",
        f"| Precision | {p_val:.4f} |",
        f"| Recall    | {r_val:.4f} |",
        f"| F1 Score  | {f1_val:.4f} |",
    ]

display(Markdown('\n'.join(report_lines)))

---
## 13 · Interactive Threshold Experiment

In [ ]:
def run_experiment(contamination):
    model = IsolationForest(n_estimators=200, contamination=contamination,
                            random_state=RANDOM_STATE, n_jobs=-1)
    preds = model.fit_predict(X_scaled)
    labels = (preds == -1).astype(int)
    n_anom = labels.sum()
    pct = n_anom / len(labels) * 100

    summary = {
        'Contamination': contamination,
        'Anomalies': n_anom,
        'Anomaly %': round(pct, 3),
    }

    if y_true is not None and set(np.unique(y_true)).issubset({0, 1}):
        summary['Precision'] = round(precision_score(y_true, labels, zero_division=0), 4)
        summary['Recall']    = round(recall_score(y_true, labels, zero_division=0), 4)
        summary['F1 Score']  = round(f1_score(y_true, labels, zero_division=0), 4)

    print(f"\n{'='*50}")
    for k, v in summary.items():
        print(f"  {k:<16}: {v}")
    print(f"{'='*50}")

    pca_e = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca_e = pca_e.fit_transform(X_scaled)
    var_e = pca_e.explained_variance_ratio_ * 100

    fig, ax = plt.subplots(figsize=(9, 5))
    for lbl, color, lbl_str, a, sz in [
        (0, PALETTE['normal'], 'Normal', 0.1, 8),
        (1, PALETTE['anomaly'], 'Anomaly', 0.7, 30),
    ]:
        m = labels == lbl
        ax.scatter(X_pca_e[m, 0], X_pca_e[m, 1], c=color, s=sz, alpha=a, label=lbl_str)
    ax.set_title(f'PCA 2D — Contamination = {contamination} → {n_anom:,} anomalies ({pct:.2f}%)', fontweight='bold')
    ax.set_xlabel(f'PC1 ({var_e[0]:.1f}%)')
    ax.set_ylabel(f'PC2 ({var_e[1]:.1f}%)')
    ax.legend()
    plt.tight_layout()
    plt.show()


cont_slider = widgets.FloatSlider(
    value=0.01,
    min=0.001,
    max=0.20,
    step=0.005,
    description='Contamination:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='55%')
)

run_btn = widgets.Button(
    description='▶ Run Experiment',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output(wait=True)
        run_experiment(cont_slider.value)

run_btn.on_click(on_click)

display(widgets.VBox([cont_slider, run_btn, output]))
run_experiment(CONTAMINATION)

---

## ✅ Pipeline Complete

All 13 sections have executed successfully. The notebook is fully reproducible — re-run all cells from top to bottom on any compatible tabular dataset.